# Azenta a4S

| Summary | Photo |
|---|---|
| - [OEM Link](https://www.azenta.com/products/automated-roll-heat-sealer-formerly-a4s)<br>- **Communication Protocol / Hardware**: Serial / USB-A<br>- **Communication Level**: Firmware (documentation shared by OEM)<br>- **Sealing Method**: Thermal (heat + pressure)<br>- **Compressed Air Required?**: No<br>- **Typical Seal Time**: ~7 seconds<br><br>The a4S has two programmatically-accessible action parameters for sealing: temperature and sealing duration. | ![a4s](img/azenta_a4s.png) |

## Setup

The a4S exposes a [Sealer](../../capabilities/sealing) and a [Temperature Controller](../../capabilities/temperature-control).

Identify the serial port on your control PC and create an `A4S` instance:

In [ ]:
from pylabrobot.azenta import A4S

s = A4S(name="a4s", port="/dev/tty.usbserial-0001")
await s.setup()

```{note}
When the a4S is first powered on, it will open its loading tray -- the **machine default state is open**.

If this is the first time you are using the a4S, follow the OEM's instructions to load a foil/film roll using the required metal film loading tool.
```

## Sealing

The a4S exposes a {class}`~pylabrobot.capabilities.sealing.sealing.Sealer` on `s.sealer`. For the full API, see [Sealing](../../capabilities/sealing).

Seal a plate with a single command:

In [ ]:
await s.sealer.seal(
    temperature=180,  # degrees Celsius
    duration=5,       # seconds
)

This command will:

1. Set the temperature
2. Wait until the temperature is reached
3. Move the plate into the machine / close the loading tray
4. Cut the film off its roll
5. Seal the film onto the plate for the specified duration
6. Move the plate out of the machine / open the loading tray

You can also open and close the loading tray independently:

In [ ]:
await s.sealer.close()
await s.sealer.open()

```{warning}
Closing the loading tray also **cuts the film/foil** without performing a seal. A single leaf of film will fall onto the tray (or onto a plate if one is present). Opening afterwards will require manual removal of that leaf.

This is a mechanical safety feature of the a4S design -- without it the film could buckle and stick to hot internals.
```

## Temperature control

The a4S exposes a {class}`~pylabrobot.capabilities.temperature_controlling.temperature_controller.TemperatureController` on `s.tc`. For the full API, see [Temperature Control](../../capabilities/temperature-control).

Pre-set the temperature to accelerate subsequent sealing steps:

In [ ]:
await s.tc.set_temperature(170)

In [ ]:
current = await s.tc.request_current_temperature()
print(f"{current:.1f} °C")

In [ ]:
await s.tc.deactivate()

## Querying machine status

The a4S driver exposes detailed status information including sensor states:

In [ ]:
status = await s.driver.request_status()
print("current_temperature:        ", status.current_temperature)
print("system_status:              ", status.system_status)
print("heater_block_status:        ", status.heater_block_status)
print("error_code:                 ", status.error_code)
print("warning_code:               ", status.warning_code)
print("sensor_status:")
print("  shuttle_middle_sensor:    ", status.sensor_status.shuttle_middle_sensor)
print("  shuttle_open_sensor:      ", status.sensor_status.shuttle_open_sensor)
print("  shuttle_close_sensor:     ", status.sensor_status.shuttle_close_sensor)
print("  clean_door_sensor:        ", status.sensor_status.clean_door_sensor)
print("  seal_roll_sensor:         ", status.sensor_status.seal_roll_sensor)
print("  heater_motor_up_sensor:   ", status.sensor_status.heater_motor_up_sensor)
print("  heater_motor_down_sensor: ", status.sensor_status.heater_motor_down_sensor)
print("remaining_time:             ", status.remaining_time)

## Teardown

In [ ]:
await s.stop()